# GPT

In [3]:
# Mathematical trick for the self-attention
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

## Bag of Words approach

In [4]:
# We want to x[b,t] = mean_{i<==t} x[b,1]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xbow[b,t] = torch.mean(x[b,:t+1], 0)

In [5]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [6]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

## Trick to find out the Bag-Of-Words quicker

In [7]:
torch.manual_seed(42)
#a = torch.ones(3,3)
a = torch.tril(torch.ones(3,3))
a = a/torch.sum(a,1,keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b

print('a==')
print(a)
print('---')
print('b==')
print(b)
print('---')
print('c==')
print(c)


a==
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
---
b==
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
---
c==
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


## Use the trick to find out the average

In [8]:
wei = torch.tril(torch.ones(8,8))
wei = wei/torch.sum(wei,1,keepdim=True)
xbow2 = wei @ x # (B,T,T) @ (B,T,C) -> (B,T,C)
torch.allclose(xbow, xbow2)

True

In [9]:
# Verison 3
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei,dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

In [ ]:
#Version 4: self-attention with one head
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# single head of self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias= False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)       # (B,T,16)
q = query (x)    # (B,T,16)
wei = q @ k.transpose(-2 , -1)    # (B,T,16) @ (B,16,T) = (B,T,T)

tril = torch.tril(torch.ones(T,T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

out.shape

# Explaing this could be a very good video.

torch.Size([4, 8, 16])

Attenion is a communication mechanism. Where number of notes in a directed graphs. Every node has some vector of information and it gets to aggregate information via a weighted sum from all the nodes that points to it.
This is dependent on the data.

8 nodes, because block size is 8.
1st node is only pointed to itself.
The 2nd node is pointed to by 1st node and itself.
...
8th node that is pointed by all its predecessors.
Communication can be applied to any arbitrary directed graph.


Note 2:
There is no notion of space. Attention simply acts over a set of vectors in this graph. So by default these nodes have no idea where they are positioned in the space. That is why we need to encode them positionally i.e. give them some information that is anchored to a specific position so that they know where they are.

In CNN, it is different, there is specific layout of the information in space and convolutional filter acts in space and the convolutional filter act in space. This is not like so in attention.

In attention there is set of vector out in space they communicate and if you want them to have a notion of space you need to specifically add it, which is what we did when we calculated positional encoding and added that information to the vectors.

The elements across the batch dimention which are independent examples never talk to each other and are always processes independetly.
![Self-Attentio](images/self-attention.png)

Current constraing is that the nodes are not talking to each other, only in the batches they are talking.

However, we might want them to talk to each other for. example. in the case of sentiment analysis. 

Encoder block: That means we need to have an encoder block and that means we do not have a triangular matrix. So comment the line wei = wei.masked_fill(tril == 0, float('-inf')).

Decoder block: Decoding language and auto-regressive format where you have to mask the Triangular  Matrix so that the nodes from the future do not talk to the past because then they will give away the answer.
I decoder block you will keep the triangular matrix that is achieved by the code wei = wei.masked_fill(tril == 0, float('-inf'))


In [42]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2197, 0.7803, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9101, 0.0837, 0.0062, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6532, 0.0849, 0.0874, 0.1745, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1780, 0.2918, 0.1833, 0.1891, 0.1576, 0.0000, 0.0000, 0.0000],
        [0.2269, 0.1897, 0.0233, 0.4739, 0.0671, 0.0192, 0.0000, 0.0000],
        [0.2066, 0.1353, 0.0754, 0.0623, 0.3932, 0.0541, 0.0731, 0.0000],
        [0.5303, 0.0405, 0.0277, 0.0424, 0.0033, 0.0579, 0.1307, 0.1672]],
       grad_fn=<SelectBackward>)